# WSMTE — 06_evaluation.ipynb
**LOCAL PC** | All metrics, statistical tests, and figure generation.

**Prerequisites:** All files downloaded from Kaggle sessions:
- `ablation/ablation_results_AE.csv`
- `ablation/ablation_results_F.csv`
- `results/saved_models/config_f_best.keras`
- `results/figures/shap_summary.png` (already generated by notebook 05)
- `data/processed/X_test.npy`, `y_clf_test.npy`, `y_reg_test.npy`
- `data/processed/final_dataset.csv`

In [ ]:
# ── Cell 1: Imports + paths ────────────────────────────────────────────────
import sys, os

# Walk up from notebook location to find project root (contains config/)
_here = os.path.abspath('.')
PROJECT_ROOT = _here
for _ in range(5):
    if os.path.isdir(os.path.join(PROJECT_ROOT, 'config')):
        break
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for local PC
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
from scipy import stats

from config.config import CONFIG

# Make all CONFIG paths absolute
for key in ['figures_dir', 'tables_dir', 'models_dir', 'results_dir',
            'processed_data_dir', 'kotekar_sentiment_file']:
    CONFIG[key] = os.path.join(PROJECT_ROOT, CONFIG[key])
CONFIG['ablation_results'] = os.path.join(PROJECT_ROOT, CONFIG['ablation_results'])

# Ensure output directories exist
for d in [CONFIG['figures_dir'], CONFIG['tables_dir'], CONFIG['models_dir']]:
    os.makedirs(d, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Figures dir  : {CONFIG["figures_dir"]}')
print(f'Tables dir   : {CONFIG["tables_dir"]}')
print('Imports OK')

## Step 1 — Merge Ablation Results (A–F)

In [ ]:
# ── Cell 2: Load + merge AE and F results ─────────────────────────────────
df_ae = pd.read_csv(os.path.join(PROJECT_ROOT, 'ablation/ablation_results_AE.csv'))
df_f  = pd.read_csv(os.path.join(PROJECT_ROOT, 'ablation/ablation_results_F.csv'))

print(f'Results A-E shape : {df_ae.shape}  configs: {df_ae["config"].unique().tolist()}')
print(f'Results F shape   : {df_f.shape}   configs: {df_f["config"].unique().tolist()}')

ablation = pd.concat([df_ae, df_f], ignore_index=True)
ablation.to_csv(CONFIG['ablation_results'], index=False)

print(f'\nMerged -> {CONFIG["ablation_results"]}  shape: {ablation.shape}')
print(ablation.groupby('config')['run'].count().rename('n_runs').to_string())

## Step 2 — Per-Config Summary Statistics

In [3]:
# ── Cell 3: Mean / Max / Std per config ───────────────────────────────────
summary_cols = ['accuracy', 'balanced_accuracy', 'auc', 'precision', 'recall', 'f1']
reg_cols = ['rmse', 'mae', 'r2']

summary = ablation.groupby('config').agg(
    mean_acc=('accuracy', 'mean'),
    max_acc=('accuracy', 'max'),
    std_acc=('accuracy', 'std'),
    mean_bacc=('balanced_accuracy', 'mean'),
    mean_auc=('auc', 'mean'),
    mean_f1=('f1', 'mean'),
    mean_rmse=('rmse', 'mean'),
    n_runs=('run', 'count'),
).round(4)
print(summary.to_string())

summary.to_csv(os.path.join(ROOT,CONFIG['tables_dir']) + 'ablation_summary.csv')
print(f"\nSaved -> {os.path.join(ROOT, CONFIG['tables_dir'], 'ablation_summary.csv')}")

        mean_acc  max_acc  std_acc  mean_bacc  mean_auc  mean_f1  mean_rmse  n_runs
config                                                                             
A         0.5654   0.5769   0.0365     0.5017    0.5016   0.6835        NaN      10
B         0.5737   0.5769   0.0069     0.4980    0.4923   0.7281        NaN      10
C         0.5782   0.5833   0.0027     0.5015    0.5021   0.7323        NaN      10
D         0.5776   0.5833   0.0020     0.5008    0.5098   0.7320        NaN      10
E         0.5750   0.5769   0.0061     0.4987    0.5021   0.7297        NaN      10
F            NaN      NaN      NaN        NaN       NaN      NaN     0.2620      10
G         0.5083   0.5897   0.0425     0.5341    0.5527   0.4322     0.2591      10
H         0.5541   0.6538   0.0620     0.5664    0.6072   0.5176     0.1911      30

Saved -> d:\WSMTE\results/tables/ablation_summary.csv


## Step 3 — Wilcoxon Signed-Rank Test: Config D vs Config F

In [ ]:
# ── Cell 4: Wilcoxon test (D vs F — single-task vs PSO final model) ────────
acc_d = ablation[ablation['config'] == 'D']['accuracy'].values
acc_f = ablation[ablation['config'] == 'F']['accuracy'].values

# Use min(len) for paired test (both should be 30)
n_pairs = min(len(acc_d), len(acc_f))
stat, p_value = stats.wilcoxon(acc_d[:n_pairs], acc_f[:n_pairs])

print(f'Wilcoxon signed-rank test: Config D vs Config F')
print(f'  n pairs     : {n_pairs}')
print(f'  Config D    : mean={acc_d.mean():.4f}  std={acc_d.std():.4f}')
print(f'  Config F    : mean={acc_f.mean():.4f}  std={acc_f.std():.4f}')
print(f'  W statistic : {stat:.4f}')
print(f'  p-value     : {p_value:.6f}')
print(f'  Significant : {"YES" if p_value < 0.05 else "NO"} (alpha=0.05)')

wilcoxon_result = {
    'test': 'Wilcoxon signed-rank',
    'config_a': 'D', 'config_b': 'F',
    'n_pairs': n_pairs,
    'mean_D': round(float(acc_d.mean()), 4),
    'mean_F': round(float(acc_f.mean()), 4),
    'W_statistic': round(float(stat), 4),
    'p_value': round(float(p_value), 6),
    'significant_at_0_05': bool(p_value < 0.05),
}
with open(os.path.join(CONFIG['results_dir'], 'wilcoxon_result.json'), 'w') as f:
    json.dump(wilcoxon_result, f, indent=2)
print(f"Saved -> {os.path.join(CONFIG['results_dir'], 'wilcoxon_result.json')}")

## Step 4 — Ablation Comparison Bar Chart

In [ ]:
# ── Cell 5: ablation_comparison.png ───────────────────────────────────────
config_order = ['A', 'B', 'C', 'D', 'E', 'F']
summary = ablation.groupby('config').agg(
    mean_acc=('accuracy', 'mean'),
    max_acc=('accuracy', 'max'),
    std_acc=('accuracy', 'std'),
    mean_bacc=('balanced_accuracy', 'mean'),
    mean_auc=('auc', 'mean'),
    mean_f1=('f1', 'mean'),
    mean_rmse=('rmse', 'mean'),
    n_runs=('run', 'count'),
).round(4)
print(summary.to_string())
summary.to_csv(os.path.join(CONFIG['tables_dir'], 'ablation_summary.csv'))
print(f"\nSaved -> {os.path.join(CONFIG['tables_dir'], 'ablation_summary.csv')}")

means = [summary.loc[c, 'mean_acc'] if c in summary.index else 0 for c in config_order]
stds  = [summary.loc[c, 'std_acc']  if c in summary.index else 0 for c in config_order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(config_order, means, yerr=stds, capsize=5,
              color=['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#E8735A'],
              alpha=0.85, edgecolor='black', linewidth=0.6)

ax.axhline(0.5853, color='red', linestyle='--', linewidth=1.2,
           label='Kotekar baseline (0.5853)')

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{m:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Ablation Config')
ax.set_ylabel('Mean Test Accuracy')
ax.set_title('WSMTE Ablation Study — Mean Accuracy per Config (±1 std)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'ablation_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(CONFIG['figures_dir'], 'ablation_comparison.png')}")

## Step 5 — Load Best Config F Model + Test Predictions

In [ ]:
# ── Cell 6: Load Config F model + test data ───────────────────────────────
import tensorflow as tf
from src.models.wsmte import build_wsmte_pso

X_test     = np.load(os.path.join(CONFIG['processed_data_dir'], 'X_test.npy'))
y_clf_test = np.load(os.path.join(CONFIG['processed_data_dir'], 'y_clf_test.npy'))
y_reg_test = np.load(os.path.join(CONFIG['processed_data_dir'], 'y_reg_test.npy'))

print(f'X_test shape     : {X_test.shape}')
print(f'y_clf_test shape : {y_clf_test.shape}  labels={set(y_clf_test.tolist())}')
print(f'y_reg_test shape : {y_reg_test.shape}')

assert X_test.shape[2] == CONFIG['n_features'], \
    f"Expected {CONFIG['n_features']} features, got {X_test.shape[2]}"
print(f'Feature count assertion passed: {X_test.shape[2]} == {CONFIG["n_features"]}')

# Load PSO weights and rebuild model — avoids WSMTEModel serialization issues
with open(os.path.join(PROJECT_ROOT, 'results', 'pso_weights.json')) as f:
    pso_info = json.load(f)
pso_w = np.array([pso_info['w_lstm'], pso_info['w_tcn'], pso_info['w_gru']])
print(f'PSO weights: LSTM={pso_w[0]:.4f}, TCN={pso_w[1]:.4f}, GRU={pso_w[2]:.4f}')

f_model = build_wsmte_pso(CONFIG, pso_w)
f_model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG['learning_rate']))
f_model.load_weights(os.path.join(CONFIG['models_dir'], 'config_f_best.keras'))
print(f'Config F model loaded OK. Trainable params: {f_model.count_params():,}')

raw_preds   = f_model(X_test, training=False)
y_reg_pred  = raw_preds[0].numpy().ravel()
y_clf_proba = raw_preds[1].numpy().ravel()
y_clf_pred  = (y_clf_proba >= 0.5).astype(int)

print(f'clf_proba range : {y_clf_proba.min():.4f} to {y_clf_proba.max():.4f}')
print(f'reg_pred  range : {y_reg_pred.min():.6f} to {y_reg_pred.max():.6f}')

## Step 6 — Confusion Matrix

In [ ]:
# ── Cell 7: confusion_matrix.png ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_clf_test, y_clf_pred)
print('Confusion Matrix (rows=actual, cols=predicted):')
print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
print(f'  FN={cm[1,0]}  TP={cm[1,1]}')

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Down (0)', 'Up (1)']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Config F — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(CONFIG['figures_dir'], 'confusion_matrix.png')}")

## Step 7 — AUC-ROC Curve

In [ ]:
# ── Cell 8: auc_roc_curve.png ─────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_clf_test, y_clf_proba)
auc_score = roc_auc_score(y_clf_test, y_clf_proba)
print(f'AUC-ROC: {auc_score:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=2, color='#4C72B0', label=f'Config F (AUC = {auc_score:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Config F (Test Set)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'auc_roc_curve.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(CONFIG['figures_dir'], 'auc_roc_curve.png')}")

In [ ]:
# ── Cell 8b: SHAP analysis (re-run locally with all features) ─────────────
from src.evaluation.shap_analysis import run_shap_analysis

X_train_shap = np.load(os.path.join(CONFIG['processed_data_dir'], 'X_train.npy'))

shap_values = run_shap_analysis(
    model=f_model,
    X_test=X_test,
    feature_names=CONFIG['feature_columns'],
    X_train=X_train_shap,
    save_path=os.path.join(CONFIG['figures_dir'], 'shap_summary.png'),
    background_size=100,
    max_display=CONFIG['n_features'],
)
print(f"SHAP values shape : {shap_values[0].shape if isinstance(shap_values, list) else shap_values.shape}")

## Step 8 — Training Loss Curves
> Uses per-run history from the best Config H run (reconstructed from ablation CSV).

In [ ]:
# ── Cell 9: Retrain Config F best seed to capture loss history ────────────
from src.models.wsmte import build_wsmte_pso
from src.training.callbacks import get_callbacks

# Load PSO weights
with open(os.path.join(PROJECT_ROOT, 'results', 'pso_weights.json')) as f:
    pso_info = json.load(f)
pso_w = np.array([pso_info['w_lstm'], pso_info['w_tcn'], pso_info['w_gru']])
print(f'PSO weights: LSTM={pso_w[0]:.4f}, TCN={pso_w[1]:.4f}, GRU={pso_w[2]:.4f}')

# Best seed from Config F results
f_df = ablation[ablation['config'] == 'F']
best_seed_val = int(f_df.loc[f_df['accuracy'].idxmax(), 'seed'])
print(f'Best Config F seed: {best_seed_val}  acc={f_df["accuracy"].max():.4f}')

tf.random.set_seed(best_seed_val)
np.random.seed(best_seed_val)

X_train = np.load(os.path.join(CONFIG['processed_data_dir'], 'X_train.npy'))
X_val   = np.load(os.path.join(CONFIG['processed_data_dir'], 'X_val.npy'))
y_reg_train = np.load(os.path.join(CONFIG['processed_data_dir'], 'y_reg_train.npy'))
y_clf_train = np.load(os.path.join(CONFIG['processed_data_dir'], 'y_clf_train.npy'))
y_reg_val   = np.load(os.path.join(CONFIG['processed_data_dir'], 'y_reg_val.npy'))
y_clf_val   = np.load(os.path.join(CONFIG['processed_data_dir'], 'y_clf_val.npy'))

lc_model = build_wsmte_pso(CONFIG, pso_w)
lc_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate'])
)
history = lc_model.fit(
    X_train, [y_reg_train, y_clf_train],
    validation_data=(X_val, [y_reg_val, y_clf_val]),
    epochs=CONFIG['pso_finetune_epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=get_callbacks(CONFIG, run_id='loss_curve_run'),
    verbose=0,
)
print(f'Loss curve training done. Epochs run: {len(history.history["loss"])}')

In [ ]:
# ── Cell 10: Plot loss curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
epochs = range(1, len(history.history['loss']) + 1)
ax.plot(epochs, history.history['loss'],     label='Train loss', lw=2)
ax.plot(epochs, history.history['val_loss'], label='Val loss',   lw=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MTL fixed-weight: 0.3×MSE + 0.7×BCE)')
ax.set_title('Config F — Training vs Validation Loss')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'loss_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(CONFIG['figures_dir'], 'loss_curves.png')}")

## Step 9 — Granger Causality Tests

In [ ]:
# ── Cell 11: Granger causality (article-level features + market, lags 1-10) ─
from src.evaluation.granger_test import run_granger_tests

# Load final_dataset.csv — has raw Close and polarity_market columns
final_df = pd.read_csv(os.path.join(ROOT, CONFIG['processed_data_dir'] + 'final_dataset.csv'))
final_df = final_df.sort_values('date').reset_index(drop=True)

print(f'final_dataset shape: {final_df.shape}')
print(f'Columns: {list(final_df.columns)}')

# Load kotekar_sentiment.csv — article-level, one row per article
kotekar_art = pd.read_csv(os.path.join(ROOT, CONFIG['kotekar_sentiment_file']))
kotekar_art['date'] = pd.to_datetime(kotekar_art['date']).dt.date
print(f'kotekar_sentiment shape: {kotekar_art.shape}')
print(f'kotekar_sentiment columns: {list(kotekar_art.columns)}')

granger_df = run_granger_tests(
    df=final_df,
    kotekar_art=kotekar_art,
    max_lag=CONFIG['granger_max_lag'],     # 10
    save_path=os.path.join(ROOT, CONFIG['tables_dir'], 'granger_results.csv'),
)
print('\n')
print(granger_df.to_string())

## Step 10 — Trading Simulation (Long-Only, Kotekar Algorithm 1)

In [13]:
# ── Cell 12: Trading simulation ───────────────────────────────────────────
from src.evaluation.trading_sim import run_trading_simulation

# Actual next-day log returns for test period
# final_dataset rows correspond to test set (after split)
n_total = len(final_df)
test_start_idx = int(n_total * (CONFIG['train_ratio']) + CONFIG['val_ratio'])
test_df = final_df.iloc[test_start_idx:].reset_index(drop=True)

# Day-over-day returns for test set (aligned to predictions)
actual_returns = test_df['Close'].pct_change().fillna(0).values
# Trim to exactly match prediction count (156 windows)
actual_returns = actual_returns[-len(y_clf_proba):]


print(f'Test predictions shape: {y_clf_proba.shape}')
print(f'Actual returns shape  : {actual_returns.shape}')

sim_results = run_trading_simulation(
    y_pred_proba=y_clf_proba,
    actual_returns=actual_returns,
    risk_free_rate=CONFIG['risk_free_rate'],
   save_path=os.path.join(ROOT, CONFIG['figures_dir'], 'trading_simulation.png'))


print('\nTrading simulation results:')
for k, v in sim_results.items():
    if isinstance(v, float):
        print(f'  {k:25s}: {v:.4f}')
    else:
        print(f'  {k:25s}: {v}')

Test predictions shape: (156,)
Actual returns shape  : (156,)

── Trading Simulation Summary ──────────────────────
  Strategy total return : 25.75%
  Buy & Hold return     : 15.47%
  Strategy Sharpe ratio : 3.2580
  Buy & Hold Sharpe     : 1.5988
  Long days             : 108
  Flat days             : 48
────────────────────────────────────────────────────
Trading simulation plot saved → d:\WSMTE\results/figures/trading_simulation.png

Trading simulation results:
  strategy_total_return    : 0.2575
  buyhold_total_return     : 0.1547
  strategy_sharpe          : 3.2580
  buyhold_sharpe           : 1.5988
  n_long_days              : 108
  n_flat_days              : 48
  strategy_cum_returns     : [ 0.0090968   0.01532363  0.01532363  0.01532363  0.01532363  0.01532363
  0.01532363  0.01292867  0.01292867 -0.00059424 -0.00059424 -0.01443907
 -0.00450885 -0.00450885 -0.00450885 -0.00450885  0.00304546  0.00304546
  0.01249405  0.0122306   0.0122306   0.0122306   0.01379884  0.01474875
 

In [14]:
# ── Cell 13: Save trading results table ───────────────────────────────────
trading_row = {k: v for k, v in sim_results.items() if not hasattr(v, '__len__')}
pd.DataFrame([trading_row]).to_csv(
    os.path.join(ROOT,CONFIG['tables_dir'] )+ 'trading_results.csv', index=False)
print(f"Saved -> {os.path.join(ROOT, CONFIG['tables_dir'], 'trading_results.csv')}")

Saved -> d:\WSMTE\results/tables/trading_results.csv


## Step 11 — Optional: Wavelet Denoising Visualisation

In [15]:
# ── Cell 14: wavelet_denoising.png (optional) ─────────────────────────────
import pywt
from src.data.preprocessor import coif3_denoise

final_df2 = pd.read_csv(os.path.join(ROOT, CONFIG['processed_data_dir'], 'final_dataset.csv'))
raw_close = final_df2['Close_d']  # denoised is in final_dataset
# We need the original close — load merged_data
merged = pd.read_csv(os.path.join(ROOT, CONFIG['processed_data_dir'], 'merged_data.csv'))
merged = merged.dropna(subset=['Close']).reset_index(drop=True)

raw_prices = merged['Close'].values[:200].copy()
denoised   = coif3_denoise(raw_prices, CONFIG)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw_prices, alpha=0.7, label='Raw Close', lw=1)
ax.plot(denoised,   alpha=0.9, label='Coif3 Denoised', lw=1.5)
ax.set_xlabel('Trading Day')
ax.set_ylabel('Nifty 50 Index')
ax.set_title('Coif3 Wavelet Denoising — Close Price (first 200 days)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, CONFIG['figures_dir'], 'wavelet_denoising.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(ROOT, CONFIG['figures_dir'], 'wavelet_denoising.png')}")

Saved -> d:\WSMTE\results/figures/wavelet_denoising.png


C:\Users\seera\AppData\Local\Temp\ipykernel_32936\3597311398.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final Summary

In [ ]:
# ── Cell 15: Summary of all outputs ──────────────────────────────────────
figures = [
    'loss_curves.png', 'confusion_matrix.png', 'auc_roc_curve.png',
    'ablation_comparison.png', 'trading_simulation.png',
    'shap_summary.png', 'wavelet_denoising.png',
]
tables = [
    'ablation_summary.csv', 'granger_results.csv', 'trading_results.csv'
]

print('=' * 55)
print('NOTEBOOK 06 COMPLETE')
print('=' * 55)
print('FIGURES:')
for fn in figures:
    path = os.path.join(CONFIG['figures_dir'], fn)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fn:35s}  {status}')

print('\nTABLES:')
for fn in tables:
    path = os.path.join(CONFIG['tables_dir'], fn)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fn:35s}  {status}')

print(f'\nablation_results.csv: {CONFIG["ablation_results"]}')
df_check = pd.read_csv(CONFIG['ablation_results'])
print(df_check.groupby('config')['accuracy'].agg(['mean','max','std']).round(4).to_string())